# LeetCode 1884 — Egg Drop With 2 Eggs and N Floors bằng Python

Link đề: https://leetcode.com/problems/egg-drop-with-2-eggs-and-n-floors/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def twoEggDrop(self, n: int) -> int:
        ...
```

Lộ trình:
1. **Minimax đệ quy theo đoạn tầng chưa biết** với số trứng còn lại — định nghĩa trực tiếp, có ghi nhớ.
2. **DP bottom-up `dp[e][k]`** cho bài trứng tổng quát với `e = 2` (`O(n²)`).
3. **Đảo bài toán**: với `m` lần thả và 2 quả trứng, kiểm tra được tối đa bao nhiêu tầng? Đáp án `m(m+1)/2` ⇒ tìm `m` nhỏ nhất (`O(√n)`).
4. **Công thức đóng** bằng căn bậc hai nguyên `math.isqrt` (`O(1)`).
5. Bonus: **dựng chiến lược** (danh sách tầng thả quả thứ nhất) và **kiểm tra bằng đối thủ** — mô phỏng mọi giá trị `f` để chắc chắn số lần thả tệ nhất đúng bằng đáp án.

## 1. Mục tiêu học

1. Hiểu mô hình **trò chơi với đối thủ** (adversary): ta chọn tầng thả, "tự nhiên" chọn kết quả vỡ/không vỡ xấu nhất ⇒ tối thiểu hóa **trường hợp tệ nhất**.
2. Viết **quy hoạch động minimax** `f(e, k) = 1 + min_x max(f(e − 1, x − 1), f(e, k − x))` và hiểu từng thành phần.
3. **Đảo bài toán** "ít lần thả nhất cho `n` tầng" thành "nhiều tầng nhất với `m` lần thả" — thường đơn giản hơn nhiều.
4. Rút ra công thức `m(m+1)/2 ≥ n` và tính bằng `isqrt` không sai số số thực.
5. Kiểm chứng một **chiến lược** cụ thể bằng cách mô phỏng mọi `f` có thể.

## 2. Đề bài & ví dụ

Bạn có **hai quả trứng giống nhau** và một tòa nhà `n` tầng đánh số `1..n`.

Tồn tại một tầng `f` (`0 ≤ f ≤ n`) sao cho: thả trứng từ tầng **cao hơn** `f` thì vỡ; thả từ tầng **`≤ f`** thì không vỡ.

Mỗi lượt, bạn lấy một quả trứng chưa vỡ và thả từ tầng `x` bất kỳ (`1 ≤ x ≤ n`). Nếu vỡ, quả đó không dùng được nữa; nếu không vỡ, có thể dùng lại.

Trả về **số lượt ít nhất** để xác định **chắc chắn** giá trị `f`.

**Ví dụ 1:** `n = 2` → `2`. Thả quả 1 ở tầng 1: vỡ ⇒ `f = 0`. Nếu không vỡ, thả ở tầng 2: vỡ ⇒ `f = 1`, không vỡ ⇒ `f = 2`.

**Ví dụ 2:** `n = 100` → `14`. Một chiến lược tối ưu: thả quả 1 ở các tầng `14, 27, 39, 50, 60, 69, 77, 84, 90, 95, 99, 100` (bản đề dùng `9, 22, 34, …` — cũng tối ưu); khi quả 1 vỡ, dùng quả 2 thử từng tầng từ trên tầng thả thành công gần nhất.

**Ràng buộc:** `1 ≤ n ≤ 1000`.

In [1]:
import random
import time
import math
from functools import lru_cache
from typing import List

examples = [(2, 2), (100, 14)]
for n, want in examples:
    print(f"n={n} -> kỳ vọng {want}")

n=2 -> kỳ vọng 2
n=100 -> kỳ vọng 14


## 3. Kiến thức nền

### 3.1 Trường hợp chỉ có **một** quả trứng

Với một quả và `k` tầng chưa biết, buộc phải thử **từ thấp lên cao từng tầng** (nếu nhảy cóc và trứng vỡ, ta không còn trứng để phân biệt các tầng bị bỏ qua) ⇒ cần đúng `k` lần trong trường hợp tệ nhất.

### 3.2 Minimax và trường hợp tệ nhất

"Xác định chắc chắn" nghĩa là chiến lược phải đúng với **mọi** `f`. Số lượt của chiến lược = **max** theo `f`; ta muốn chiến lược có max **nhỏ nhất**.

In [2]:
def one_egg_worst(k):
    """Một quả trứng, k tầng: thử 1, 2, ..., k. Trả về số lượt tệ nhất bằng mô phỏng mọi f."""
    worst = 0
    for f in range(0, k + 1):
        drops = 0
        for floor in range(1, k + 1):
            drops += 1
            if floor > f:        # vỡ
                break
        worst = max(worst, drops)
    return worst

print([one_egg_worst(k) for k in range(6)], "(= k)")

[0, 1, 2, 3, 4, 5] (= k)


### 3.3 `functools.lru_cache` cho DP đệ quy

In [3]:
@lru_cache(maxsize=None)
def fib(i):
    return i if i < 2 else fib(i - 1) + fib(i - 2)
print(fib(80), fib.cache_info().currsize, "trạng thái")

23416728348467685 81 trạng thái


### 3.4 Căn bậc hai nguyên `math.isqrt`

`math.isqrt(v)` trả về `⌊√v⌋` **chính xác** với số nguyên (không sai số dấu phẩy động). Dùng để giải `m(m+1)/2 ≥ n`.

In [4]:
print(math.isqrt(15), math.isqrt(16), math.isqrt(10**30 + 1))

3 4 1000000000000000


### 3.5 Góc nhìn tổ hợp: `F(e, m) = Σ_{i=1..e} C(m, i)`

Một cách đếm khác cho "số tầng phủ được": mỗi tầng `f` tương ứng với một **dãy kết quả** (vỡ/không vỡ) của các lượt thả; với `e` quả trứng, dãy có tối đa `e` lần "vỡ". Số dãy độ dài `m` có từ 1 tới `e` lần vỡ (đánh dấu vị trí vỡ) là `Σ_{i=1..e} C(m, i)`. Với `e = 2`: `m + m(m−1)/2 = m(m+1)/2`. ✓

In [5]:
from math import comb
for m in range(1, 8):
    print(f"m={m}: C(m,1)+C(m,2) = {comb(m, 1) + comb(m, 2):3d} | m(m+1)/2 = {m * (m + 1) // 2:3d} | 3 trứng: {comb(m,1)+comb(m,2)+comb(m,3)}")

m=1: C(m,1)+C(m,2) =   1 | m(m+1)/2 =   1 | 3 trứng: 1
m=2: C(m,1)+C(m,2) =   3 | m(m+1)/2 =   3 | 3 trứng: 3
m=3: C(m,1)+C(m,2) =   6 | m(m+1)/2 =   6 | 3 trứng: 7
m=4: C(m,1)+C(m,2) =  10 | m(m+1)/2 =  10 | 3 trứng: 14
m=5: C(m,1)+C(m,2) =  15 | m(m+1)/2 =  15 | 3 trứng: 25
m=6: C(m,1)+C(m,2) =  21 | m(m+1)/2 =  21 | 3 trứng: 41
m=7: C(m,1)+C(m,2) =  28 | m(m+1)/2 =  28 | 3 trứng: 63


## 4. Ý tưởng nền tảng / trực giác

Nếu thả quả thứ nhất ở tầng `x`:
- **Vỡ** ⇒ `f ∈ [0, x − 1]`, còn **một** quả ⇒ cần thêm `x − 1` lượt (thử `1..x−1`).
- **Không vỡ** ⇒ `f ∈ [x, n]` — giống bài toán ban đầu với `n − x` tầng phía trên, vẫn **hai** quả.

Ví dụ `n = 10`. Chiến lược "chia đều" thả ở `5`: nếu vỡ cần `1 + 4 = 5`; không vỡ còn 5 tầng… Chiến lược tốt hơn: **giảm dần khoảng cách** — thả ở `4, 7, 9, 10`:
- vỡ ở `4`: `1 + 3 = 4` lượt;
- vỡ ở `7`: `2 + 2 = 4` lượt (thử 5, 6);
- vỡ ở `9`: `3 + 1 = 4` lượt (thử 8);
- …mọi trường hợp `≤ 4`.

Mỗi lần quả 1 sống sót, ta "tiêu" một lượt, nên khoảng cho quả 2 ở bước sau phải **ngắn đi 1** để tổng không đổi ⇒ các bước nhảy `m, m − 1, m − 2, …, 1` phủ `m(m+1)/2` tầng.

In [6]:
def simulate(strategy, n):
    """strategy: danh sách tầng thả quả 1 (tăng dần, phần tử cuối = n). Trả về số lượt tệ nhất trên mọi f."""
    worst = 0
    for f in range(0, n + 1):
        drops = 0; low = 1
        for x in strategy:
            drops += 1
            if x > f:                                    # quả 1 vỡ ở x -> quả 2 thử low..x-1
                for y in range(low, x):
                    drops += 1
                    if y > f:
                        break
                break
            low = x + 1
        worst = max(worst, drops)
    return worst

print("n=10, thả ở [5,10]:", simulate([5, 10], 10))
print("n=10, thả ở [4,7,9,10]:", simulate([4, 7, 9, 10], 10))
print("n=100, chiến lược trong đề:", simulate([9, 22, 34, 45, 55, 64, 72, 79, 85, 90, 94, 97, 99, 100], 100))

n=10, thả ở [5,10]: 6
n=10, thả ở [4,7,9,10]: 4
n=100, chiến lược trong đề: 14


## 5. Cách 1 — Brute Force / baseline: minimax đệ quy có ghi nhớ

**Trạng thái:** `g(e, k)` = số lượt ít nhất để xác định `f` khi còn `e` quả trứng và `k` tầng chưa biết (các tầng chưa biết luôn là một đoạn liên tiếp, nên chỉ số lượng `k` quan trọng).

**Chuyển:** `g(e, 0) = 0`; `g(1, k) = k`; với `e = 2`, `k ≥ 1`:

`g(2, k) = 1 + min_{1 ≤ x ≤ k} max( g(1, x − 1), g(2, k − x) )`.

`max` là "đối thủ chọn kết quả xấu nhất", `min` là "ta chọn tầng tốt nhất".

In [7]:
@lru_cache(maxsize=None)
def g(e, k):
    if k == 0:
        return 0
    if e == 1:
        return k
    return 1 + min(max(g(e - 1, x - 1), g(e, k - x)) for x in range(1, k + 1))

def two_egg_memo(n):
    return g(2, n)

for n, want in examples:
    print(two_egg_memo(n), "kỳ vọng", want)

2 kỳ vọng 2
14 kỳ vọng 14


**Độ phức tạp:** `n` trạng thái (với `e = 2`), mỗi trạng thái duyệt `O(n)` lựa chọn ⇒ `O(n²)` = `10⁶` với `n = 1000` — chạy được. Độ sâu đệ quy tới `n`: với `n = 1000` sát giới hạn đệ quy mặc định; ta sẽ dùng bottom-up cho an toàn.

Một brute force **thuần** (không ghi nhớ) có số lời gọi tăng theo hàm mũ — minh họa:

In [8]:
def g_no_memo(e, k, counter):
    counter[0] += 1
    if k == 0: return 0
    if e == 1: return k
    return 1 + min(max(g_no_memo(e - 1, x - 1, counter), g_no_memo(e, k - x, counter)) for x in range(1, k + 1))

for k in [8, 12, 16]:
    c = [0]; r = g_no_memo(2, k, c)
    print(f"k={k}: kết quả {r}, số lời gọi không ghi nhớ = {c[0]}")

k=8: kết quả 4, số lời gọi không ghi nhớ = 511
k=12: kết quả 5, số lời gọi không ghi nhớ = 8191
k=16: kết quả 6, số lời gọi không ghi nhớ = 131071


**Nút thắt:** ngay cả DP `O(n²)` cũng "thừa" — cấu trúc bài cho phép tính trực tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Đảo bài toán.** Gọi `F(m)` = số tầng **nhiều nhất** có thể xử lý với `m` lượt thả và 2 quả trứng (xác định được `f` chắc chắn).

- Lượt đầu, thả quả 1 ở tầng `x`. Nếu vỡ, còn `m − 1` lượt với 1 quả ⇒ kiểm được tối đa `m − 1` tầng phía dưới ⇒ chọn `x = m` (tầng `1..m−1` dưới nó).
- Nếu không vỡ, còn `m − 1` lượt với 2 quả ⇒ kiểm được `F(m − 1)` tầng phía trên.

`F(m) = m + F(m − 1)`, `F(0) = 0` ⇒ `F(m) = m(m+1)/2`.

**Đáp án:** `m` nhỏ nhất với `m(m+1)/2 ≥ n`.

**Công thức đóng:** `m(m+1)/2 ≥ n ⇔ m ≥ (√(8n+1) − 1)/2` ⇒ `m = ⌈(√(8n+1) − 1)/2⌉`. Dùng `isqrt` để tránh sai số.

**Chiến lược tương ứng:** thả quả 1 ở các tầng `m, m + (m−1), m + (m−1) + (m−2), …` (cắt ở `n`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP bottom-up `dp[e][k]`

In [9]:
def two_egg_dp(n, eggs=2):
    INF = float("inf")
    dp = [[0] * (n + 1) for _ in range(eggs + 1)]
    for k in range(n + 1):
        dp[1][k] = k
    for e in range(2, eggs + 1):
        for k in range(1, n + 1):
            best = INF
            for x in range(1, k + 1):
                worst = max(dp[e - 1][x - 1], dp[e][k - x])
                if worst < best:
                    best = worst
            dp[e][k] = 1 + best
    return dp[eggs][n]

for n, want in examples:
    print(two_egg_dp(n), "kỳ vọng", want)

2 kỳ vọng 2
14 kỳ vọng 14


Code nộp LeetCode của Cách 2 (với `e = 2`, gom thành mảng 1 chiều):

```python
class Solution:
    def twoEggDrop(self, n: int) -> int:
        dp = [0] * (n + 1)                  # dp[k] = g(2, k)
        for k in range(1, n + 1):
            dp[k] = 1 + min(max(x - 1, dp[k - x]) for x in range(1, k + 1))
        return dp[n]
```

**Độ phức tạp:** Time `O(n²)` (`e = 2`), Space `O(n)`. Bài trứng tổng quát `e` quả: `O(e · n²)`.

### 7.2 Cách 3 — Tăng `m` cho tới khi `m(m+1)/2 ≥ n`

In [10]:
def two_egg_loop(n):
    m = 0
    covered = 0
    while covered < n:
        m += 1
        covered += m           # F(m) = F(m-1) + m
    return m

for n, want in examples:
    print(two_egg_loop(n), "kỳ vọng", want)

2 kỳ vọng 2
14 kỳ vọng 14


**Độ phức tạp:** `m ≈ √(2n)` ⇒ Time `O(√n)`, Space `O(1)`.

### 7.3 Cách 4 — Công thức đóng với `isqrt`

`m = ⌈(√(8n+1) − 1)/2⌉`. Tính `r = isqrt(8n + 1)`, `m = (r − 1) // 2`; nếu `m(m+1)/2 < n` thì `m += 1` (điều chỉnh làm tròn lên).

In [11]:
def two_egg_formula(n):
    r = math.isqrt(8 * n + 1)
    m = (r - 1) // 2
    if m * (m + 1) // 2 < n:
        m += 1
    return m

for n, want in examples:
    print(two_egg_formula(n), "kỳ vọng", want)
print([two_egg_formula(n) for n in range(1, 22)])

2 kỳ vọng 2
14 kỳ vọng 14
[1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6]


**Độ phức tạp:** Time `O(1)` (với `isqrt` trên số nhỏ), Space `O(1)`.

### 7.4 Cách 5 — Dựng chiến lược và kiểm tra bằng đối thủ

Từ `m`, các tầng thả quả 1 là tổng dồn `m, m−1, m−2, …` (cắt ở `n`). Hàm `simulate` (mục 4) thử **mọi** `f ∈ [0, n]` và trả về số lượt tệ nhất — phải đúng bằng `m`.

In [12]:
def build_strategy(n):
    m = two_egg_formula(n)
    floors, cur, step = [], 0, m
    while cur < n:
        cur = min(n, cur + step)
        floors.append(cur)
        step -= 1
    return m, floors

for n in [2, 10, 36, 100]:
    m, floors = build_strategy(n)
    print(f"n={n}: m={m}, thả quả 1 ở {floors}, tệ nhất (mô phỏng) = {simulate(floors, n)}")

n=2: m=2, thả quả 1 ở [2], tệ nhất (mô phỏng) = 2
n=10: m=4, thả quả 1 ở [4, 7, 9, 10], tệ nhất (mô phỏng) = 4
n=36: m=8, thả quả 1 ở [8, 15, 21, 26, 30, 33, 35, 36], tệ nhất (mô phỏng) = 8
n=100: m=14, thả quả 1 ở [14, 27, 39, 50, 60, 69, 77, 84, 90, 95, 99, 100], tệ nhất (mô phỏng) = 14


## 8. TRACE từng bước

Trace Cách 3 với `n = 100`: cộng dồn `F(m)` cho tới khi `≥ 100`.

In [13]:
def trace_loop(n):
    m = covered = 0
    while covered < n:
        m += 1; covered += m
        print(f"m={m:2d}: F(m) = {covered:3d} tầng {'(đủ)' if covered >= n else ''}")
    return m

trace_loop(100)

m= 1: F(m) =   1 tầng 
m= 2: F(m) =   3 tầng 
m= 3: F(m) =   6 tầng 
m= 4: F(m) =  10 tầng 
m= 5: F(m) =  15 tầng 
m= 6: F(m) =  21 tầng 
m= 7: F(m) =  28 tầng 
m= 8: F(m) =  36 tầng 
m= 9: F(m) =  45 tầng 
m=10: F(m) =  55 tầng 
m=11: F(m) =  66 tầng 
m=12: F(m) =  78 tầng 
m=13: F(m) =  91 tầng 
m=14: F(m) = 105 tầng (đủ)


14

Trace Cách 2 (bảng `dp[2][k]`) cho `k = 1..10`, kèm tầng thả tốt nhất `x*`:

In [14]:
def trace_dp(n):
    dp = [0] * (n + 1)
    for k in range(1, n + 1):
        opts = [(max(x - 1, dp[k - x]), x) for x in range(1, k + 1)]
        best, xb = min(opts)
        dp[k] = 1 + best
        print(f"k={k:2d}: x* = {xb:2d} (vỡ -> {xb - 1} lượt, không vỡ -> dp[{k - xb}]={dp[k - xb]}) => dp[{k}] = {dp[k]}")
    return dp[n]

trace_dp(10)

k= 1: x* =  1 (vỡ -> 0 lượt, không vỡ -> dp[0]=0) => dp[1] = 1
k= 2: x* =  1 (vỡ -> 0 lượt, không vỡ -> dp[1]=1) => dp[2] = 2
k= 3: x* =  2 (vỡ -> 1 lượt, không vỡ -> dp[1]=1) => dp[3] = 2
k= 4: x* =  1 (vỡ -> 0 lượt, không vỡ -> dp[3]=2) => dp[4] = 3
k= 5: x* =  2 (vỡ -> 1 lượt, không vỡ -> dp[3]=2) => dp[5] = 3
k= 6: x* =  3 (vỡ -> 2 lượt, không vỡ -> dp[3]=2) => dp[6] = 3
k= 7: x* =  1 (vỡ -> 0 lượt, không vỡ -> dp[6]=3) => dp[7] = 4
k= 8: x* =  2 (vỡ -> 1 lượt, không vỡ -> dp[6]=3) => dp[8] = 4
k= 9: x* =  3 (vỡ -> 2 lượt, không vỡ -> dp[6]=3) => dp[9] = 4
k=10: x* =  4 (vỡ -> 3 lượt, không vỡ -> dp[6]=3) => dp[10] = 4


4

Bảng tóm tắt (đối chiếu `F(m) = m(m+1)/2`):

| m | F(m) | các n có đáp án m |
|---|---|---|
| 1 | 1 | 1 |
| 2 | 3 | 2–3 |
| 3 | 6 | 4–6 |
| 4 | 10 | 7–10 |
| 14 | 105 | 92–105 (gồm 100) |

Trace "đối thủ" cho chiến lược `n = 10`: với mỗi `f`, in dãy lượt thả.

In [15]:
def trace_adversary(floors, n):
    for f in range(0, n + 1):
        log = []; low = 1
        for x in floors:
            broke = x > f; log.append(f"Q1@{x}{'v' if broke else 'ok'}")
            if broke:
                for y in range(low, x):
                    b2 = y > f; log.append(f"Q2@{y}{'v' if b2 else 'ok'}")
                    if b2: break
                break
            low = x + 1
        print(f"f={f:2d}: {' '.join(log)}  ({len(log)} lượt)")

trace_adversary([4, 7, 9, 10], 10)

f= 0: Q1@4v Q2@1v  (2 lượt)
f= 1: Q1@4v Q2@1ok Q2@2v  (3 lượt)
f= 2: Q1@4v Q2@1ok Q2@2ok Q2@3v  (4 lượt)
f= 3: Q1@4v Q2@1ok Q2@2ok Q2@3ok  (4 lượt)
f= 4: Q1@4ok Q1@7v Q2@5v  (3 lượt)
f= 5: Q1@4ok Q1@7v Q2@5ok Q2@6v  (4 lượt)
f= 6: Q1@4ok Q1@7v Q2@5ok Q2@6ok  (4 lượt)
f= 7: Q1@4ok Q1@7ok Q1@9v Q2@8v  (4 lượt)
f= 8: Q1@4ok Q1@7ok Q1@9v Q2@8ok  (4 lượt)
f= 9: Q1@4ok Q1@7ok Q1@9ok Q1@10v  (4 lượt)
f=10: Q1@4ok Q1@7ok Q1@9ok Q1@10ok  (4 lượt)


### 8.1 Tầng thả đầu tiên tối ưu không duy nhất

Đề gợi ý thả lần đầu ở tầng `9` cho `n = 100`, còn chiến lược "bước giảm dần" của ta thả ở `14`. Cả hai đều tối ưu! Liệt kê mọi `x` cho lượt đầu đạt `1 + max(x − 1, dp[n − x]) = dp[n]`:

In [16]:
def optimal_first_floors(n):
    d = [0] * (n + 1)
    for k in range(1, n + 1):
        d[k] = 1 + min(max(x - 1, d[k - x]) for x in range(1, k + 1))
    return [x for x in range(1, n + 1) if 1 + max(x - 1, d[n - x]) == d[n]]

for n in [10, 36, 100]:
    print(f"n={n}: các tầng thả đầu tiên tối ưu = {optimal_first_floors(n)}")

n=10: các tầng thả đầu tiên tối ưu = [4]
n=36: các tầng thả đầu tiên tối ưu = [8]
n=100: các tầng thả đầu tiên tối ưu = [9, 10, 11, 12, 13, 14]


Với `n = 100`, mọi `x ∈ [9, 14]` đều tối ưu: thả thấp hơn `14` vẫn được vì `F(14) = 105` dư `5` tầng so với `100` — ta có "khoảng trống" để chọn. Chiến lược của đề (`9, 22, 34, …`) dùng phần dư này ở đầu, còn của ta dùng ở cuối (`99, 100`).

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def twoEggDrop(self, n: int) -> int:
        r = math.isqrt(8 * n + 1)
        m = (r - 1) // 2
        if m * (m + 1) // 2 < n:
            m += 1
        return m

sol = Solution()
for n, want in examples:
    got = sol.twoEggDrop(n)
    print(got, "OK" if got == want else "SAI")

2 OK
14 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (một quả).** Với 1 quả và `k` tầng chưa biết, cần đúng `k` lượt (tệ nhất). *Cần:* nếu một chiến lược thả ở tầng `y` mà còn tầng `< y` chưa kiểm, đối thủ cho trứng vỡ ⇒ không phân biệt được các tầng đó. Nên phải thả lần lượt từ thấp lên ⇒ `k` lượt khi `f = k` hoặc `f = k − 1`. *Đủ:* chiến lược đó dùng `≤ k` lượt.

**Bổ đề 2 (`F(m) = m(m+1)/2`).** Quy nạp theo `m`.
- *Đạt được:* thả quả 1 ở tầng `m`; vỡ ⇒ `m − 1` tầng dưới, dùng quả 2 với `m − 1` lượt còn lại (Bổ đề 1); không vỡ ⇒ dùng `m − 1` lượt còn lại cho `F(m − 1)` tầng phía trên. Tổng `m + F(m − 1)`.
- *Không vượt được:* lượt đầu thả ở `x`. Nếu vỡ, còn `m − 1` lượt với 1 quả ⇒ `x − 1 ≤ m − 1`. Nếu không vỡ, phần trên `≤ F(m − 1)`. Vậy tổng số tầng `≤ (x − 1) + 1 + F(m − 1) ≤ m + F(m − 1)`.

**Kết luận.** `n` tầng xử lý được với `m` lượt ⇔ `n ≤ F(m)` ⇒ đáp án `min{m : m(m+1)/2 ≥ n}`. DP minimax (Cách 1, 2) tính cùng giá trị theo định nghĩa trực tiếp — mục 12 kiểm chứng với mọi `n ≤ 1000`.

**Công thức `isqrt`.** Đặt `r = ⌊√(8n+1)⌋` và `m₀ = ⌊(r − 1)/2⌋`. Vì `(2m₀ + 1)² ≤ r² ≤ 8n + 1` nên `m₀(m₀+1)/2 ≤ n` — tức `m₀` không vượt đáp án. Đáp án đúng là `m₀` nếu `m₀(m₀+1)/2 = n`, ngược lại là `m₀ + 1` (vì `(m₀+1)(m₀+2)/2 > n` do `r` là phần nguyên lớn nhất). Bước `if m(m+1)/2 < n: m += 1` thực hiện đúng phép chọn này; mục 12 kiểm tra với mọi `n ≤ 1000`.

## 11. Phân tích Time / Space

Gọi `n ≤ 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Minimax đệ quy + memo | `O(n²)` | `O(n)` + stack | đệ quy sâu `n` |
| 2. DP bottom-up | `O(n²)` (`O(e n²)` tổng quát) | `O(n)` | tổng quát cho `e` trứng |
| 3. Tăng `m` | `O(√n)` | `O(1)` | đơn giản |
| 4. Công thức `isqrt` | `O(1)` | `O(1)` | **nộp bài** |
| 5. Dựng + mô phỏng | `O(n · m)` | `O(m)` | kiểm chứng chiến lược |

In [18]:
for name, fn in [("dp", two_egg_dp), ("loop", two_egg_loop), ("formula", two_egg_formula), ("submit", Solution().twoEggDrop)]:
    t0 = time.perf_counter(); r = [fn(n) for n in (1, 10, 100, 1000)]
    print(f"{name:7s}: {r}  {time.perf_counter() - t0:.4f}s")

dp     : [1, 4, 14, 45]  0.0282s
loop   : [1, 4, 14, 45]  0.0000s
formula: [1, 4, 14, 45]  0.0000s
submit : [1, 4, 14, 45]  0.0000s


## 12. Bộ test

Edge case quanh các mốc `F(m)` (`n = F(m)` và `n = F(m) + 1` làm đáp án nhảy bậc), rồi kiểm tra **mọi** `n ∈ [1, 1000]` (toàn bộ miền input): công thức = vòng lặp = DP; và chiến lược dựng được đạt đúng số lượt khi mô phỏng mọi `f`.

In [19]:
for n, want in examples + [(1, 1), (3, 2), (4, 3), (6, 3), (7, 4), (105, 14), (106, 15), (300, 24)]:
    assert two_egg_memo(n) == two_egg_dp(n) == two_egg_loop(n) == two_egg_formula(n) == Solution().twoEggDrop(n) == want, n

# DP một lần cho mọi n <= 1000 (mảng 1 chiều), so với các cách nhanh
N = 1000
dp = [0] * (N + 1)
for k in range(1, N + 1):
    dp[k] = 1 + min(max(x - 1, dp[k - x]) for x in range(1, k + 1))
for n in range(1, N + 1):
    assert dp[n] == two_egg_loop(n) == two_egg_formula(n) == Solution().twoEggDrop(n), n
print("mọi n trong [1, 1000]: DP minimax = vòng lặp = công thức")

mọi n trong [1, 1000]: DP minimax = vòng lặp = công thức


In [20]:
# chiến lược dựng được có số lượt tệ nhất đúng bằng đáp án (mô phỏng mọi f)
for n in list(range(1, 60)) + [100, 250, 500, 1000]:
    m, floors = build_strategy(n)
    assert simulate(floors, n) == m, n
# và không chiến lược nào với m-1 lượt đủ: kiểm bằng F(m-1) < n
for n in range(1, 1001):
    m = two_egg_formula(n)
    assert (m - 1) * m // 2 < n <= m * (m + 1) // 2
print("chiến lược dựng được đạt đúng m lượt; F(m-1) < n <= F(m) với mọi n")

chiến lược dựng được đạt đúng m lượt; F(m-1) < n <= F(m) với mọi n


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tìm kiếm nhị phân (như có vô số trứng)

In [21]:
def wrong_binary_search(n):
    return math.ceil(math.log2(n + 1))       # SAI: chỉ đúng khi có đủ trứng để chia đôi

for n in [2, 10, 100, 1000]:
    print(f"n={n}: sai = {wrong_binary_search(n)} | đúng = {two_egg_formula(n)}")

n=2: sai = 2 | đúng = 2
n=10: sai = 4 | đúng = 4
n=100: sai = 7 | đúng = 14
n=1000: sai = 10 | đúng = 45


Với 2 quả, sau khi quả 1 vỡ ta phải dò tuyến tính — chia đôi không còn hợp lệ. **Sửa:** DP hoặc công thức `m(m+1)/2`.

### Lỗi 2 — Bước nhảy đều `√n`

In [22]:
def sqrt_strategy(n):
    s = max(1, math.isqrt(n))
    floors = list(range(s, n + 1, s))
    if not floors or floors[-1] != n:
        floors.append(n)
    return floors

for n in [10, 100, 1000]:
    print(f"n={n}: nhảy đều √n -> tệ nhất {simulate(sqrt_strategy(n), n)} | tối ưu {two_egg_formula(n)}")

n=10: nhảy đều √n -> tệ nhất 5 | tối ưu 4
n=100: nhảy đều √n -> tệ nhất 19 | tối ưu 14
n=1000: nhảy đều √n -> tệ nhất 62 | tối ưu 45


Nhảy đều cho khoảng `≈ 2√n`, tệ hơn `√(2n)` của bước giảm dần. **Sửa:** bước nhảy giảm dần `m, m−1, …`.

### Lỗi 3 — Công thức số thực bị làm tròn sai

In [23]:
def wrong_float(n):
    return int((math.sqrt(8 * n + 1) - 1) / 2)     # SAI: làm tròn XUỐNG thay vì lên

bad = [n for n in range(1, 1001) if wrong_float(n) != two_egg_formula(n)]
print("số n sai:", len(bad), "| ví dụ:", bad[:8])
print("n=100: sai =", wrong_float(100), "| đúng =", two_egg_formula(100))

số n sai: 956 | ví dụ: [2, 4, 5, 7, 8, 9, 11, 12]
n=100: sai = 13 | đúng = 14


`int(...)` cắt phần lẻ (làm tròn xuống), trong khi cần `⌈…⌉`. Ngay cả `math.ceil` trên số thực cũng tiềm ẩn sai số với `n` lớn. **Sửa:** `isqrt` + bước điều chỉnh (Cách 4) hoặc vòng lặp (Cách 3).

### Lỗi 4 — DP quên cộng 1 cho lượt thả hiện tại, hoặc dùng `min` thay `max` cho phản ứng của đối thủ

In [24]:
def wrong_dp_min(n):
    dp = [0] * (n + 1)
    for k in range(1, n + 1):
        dp[k] = 1 + min(min(x - 1, dp[k - x]) for x in range(1, k + 1))   # SAI: đối thủ chọn xấu nhất (max)
    return dp[n]

for n in [2, 10, 100]:
    print(f"n={n}: dùng min = {wrong_dp_min(n)} | đúng = {two_egg_formula(n)}")

n=2: dùng min = 1 | đúng = 2
n=10: dùng min = 1 | đúng = 4
n=100: dùng min = 1 | đúng = 14


**Sửa:** `max` (trường hợp tệ nhất do đối thủ chọn) bên trong, `min` (lựa chọn của ta) bên ngoài.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xác định **chắc chắn**, tối thiểu số lượt" ⇒ minimax: `min` (ta chọn) của `max` (đối thủ chọn).
- Bài trứng ⇒ **đảo bài toán**: "với `m` lượt và `e` trứng, phủ được bao nhiêu tầng?" — `F(e, m) = F(e−1, m−1) + 1 + F(e, m−1)`.
- Khi đáp án là `m` nhỏ nhất với `S(m) ≥ n` và `S` tăng ⇒ vòng lặp hoặc nhị phân trên `m`.
- Căn bậc hai với số nguyên ⇒ `math.isqrt`, tránh `math.sqrt` + làm tròn.
- Họ bài: 887 *Super Egg Drop* (`k` trứng, `n ≤ 10⁴`), 375 *Guess Number Higher or Lower II* (minimax trên đoạn).

## 15. Biến thể của bài

1. **887 Super Egg Drop (`k` trứng):** `F(e, m) = F(e−1, m−1) + F(e, m−1) + 1` — tìm `m` nhỏ nhất với `F(k, m) ≥ n` trong `O(k · m)`.
2. **Ba quả trứng:** `F(3, m) = C(m,1) + C(m,2) + C(m,3)`. Tổng quát `F(e, m) = Σ_{i=1..e} C(m, i)`.
3. **Trả về chiến lược** (danh sách tầng thả quả 1): Cách 5.
4. **Tầng đánh số từ 0 / tòa nhà có tầng hầm:** chỉ đổi số tầng `n`.

In [25]:
def super_egg_drop(k, n):
    """Biến thể 1: số lượt ít nhất với k trứng, n tầng (LeetCode 887)."""
    F = [0] * (k + 1)                  # F[e] = số tầng phủ được với m lượt hiện tại
    m = 0
    while F[k] < n:
        m += 1
        for e in range(k, 0, -1):
            F[e] = F[e] + F[e - 1] + 1  # F(e, m) = F(e, m-1) + F(e-1, m-1) + 1
    return m

def egg_dp_general(k, n):
    @lru_cache(maxsize=None)
    def h(e, t):
        if t == 0: return 0
        if e == 1: return t
        return 1 + min(max(h(e - 1, x - 1), h(e, t - x)) for x in range(1, t + 1))
    return h(k, n)

from math import comb
for k in range(1, 5):
    for n in range(0, 80):
        assert super_egg_drop(k, n) == egg_dp_general(k, n)
for m in range(1, 15):
    # công thức tổng tổ hợp: với m lượt, 3 trứng phủ đúng C(m,1)+C(m,2)+C(m,3) tầng
    F3m = comb(m, 1) + comb(m, 2) + comb(m, 3)
    assert super_egg_drop(3, F3m) == m and super_egg_drop(3, F3m + 1) == m + 1
print("887 khớp DP minimax với k ≤ 4, n < 80; super_egg_drop(2, 100) =", super_egg_drop(2, 100), "| (3, 100) =", super_egg_drop(3, 100))

887 khớp DP minimax với k ≤ 4, n < 80; super_egg_drop(2, 100) = 14 | (3, 100) = 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax memo | định nghĩa trực tiếp | `O(n²)` | `O(n)` | hiểu bài |
| 2. DP bottom-up | bảng `dp[e][k]` | `O(e n²)` | `O(n)` | tổng quát, kiểm chứng |
| 3. Tăng `m` | `F(m) = F(m−1) + m` | `O(√n)` | `O(1)` | đơn giản, an toàn |
| 4. `isqrt` | công thức đóng | `O(1)` | `O(1)` | **nộp bài** |
| 5. Chiến lược + đối thủ | dựng & mô phỏng | `O(n m)` | `O(m)` | chứng minh bằng máy |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `twoEggDrop(1)`, `twoEggDrop(3)`, `twoEggDrop(4)`, `twoEggDrop(1000)`?

**Bài 2 (tính tay):** dựng chiến lược tối ưu cho `n = 15` (danh sách tầng thả quả 1) và kiểm tra bằng tay trường hợp quả 1 vỡ ở lần thả thứ 3.

**Bài 3 (điền TODO):** hoàn thiện `max_floors(m)` trả về số tầng tối đa xử lý được với `m` lượt và 2 trứng — **không** dùng công thức đóng (dùng đệ quy/DP theo định nghĩa ở mục 6).

**Bài 4 (tự viết lại):** viết `two_egg_binary(n)` tìm `m` nhỏ nhất bằng **tìm kiếm nhị phân** trên `m` với hàm `F(m) = m(m+1)/2`.

**Bài 5 (mini challenge):** với **3 quả trứng**, viết công thức `F(3, m)` và hàm tìm `m` nhỏ nhất cho `n`; kiểm chứng bằng DP minimax `egg_dp_general(3, n)` với mọi `n ≤ 200`.

In [26]:
def max_floors(m):
    # TODO: F(0) = 0, F(m) = (m - 1) + 1 + F(m - 1)   (vỡ: m-1 tầng dưới; không vỡ: F(m-1) tầng trên)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `F(1) = 1, F(2) = 3, F(3) = 6, …`. (2) `n = 1 → 1`, `n = 3 → 2`, `n = 4 → 3`. (3) `F(44) = 990 < 1000 ≤ F(45) = 1035` ⇒ `45`.

**Bài 2:** (1) `m = 5` (`F(5) = 15`). (2) Tầng `5, 9, 12, 14, 15`. (3) Vỡ ở lần 3 (tầng 12) ⇒ quả 2 thử `10, 11` ⇒ tổng `3 + 2 = 5` lượt.

**Bài 3:** (1) Viết vòng lặp `for i in range(1, m + 1): F += i`. (2) Hoặc đệ quy có `lru_cache`. (3) So với `m(m+1)//2`.

**Bài 4:** (1) `lo = 0`, `hi = n` (vì `F(n) ≥ n`). (2) `if F(mid) >= n: hi = mid else lo = mid + 1`. (3) Trả về `lo`.

**Bài 5:** (1) `F(3, m) = F(2, m−1) + 1 + F(3, m−1)`. (2) Đóng: `m + m(m−1)/2 + m(m−1)(m−2)/6`. (3) Vòng lặp tăng `m`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert [Solution().twoEggDrop(n) for n in (1, 3, 4, 1000)] == [1, 2, 3, 45]

# Bài 2
m, floors = build_strategy(15)
assert (m, floors) == (5, [5, 9, 12, 14, 15]) and simulate(floors, 15) == 5

# Bài 3
@lru_cache(maxsize=None)
def max_floors_ans(m):
    return 0 if m == 0 else (m - 1) + 1 + max_floors_ans(m - 1)
assert all(max_floors_ans(m) == m * (m + 1) // 2 for m in range(0, 60))

# Bài 4
def two_egg_binary(n):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if mid * (mid + 1) // 2 >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo
assert all(two_egg_binary(n) == two_egg_formula(n) for n in range(1, 1001))
print("Bài 1-4: OK")

Bài 1-4: OK


In [28]:
# Bài 5 — ba quả trứng
def F3(m):
    return m + m * (m - 1) // 2 + m * (m - 1) * (m - 2) // 6

def three_egg_drop(n):
    m = 0
    while F3(m) < n:
        m += 1
    return m

for n in range(0, 201):
    assert three_egg_drop(n) == egg_dp_general(3, n), n
print("Bài 5: 3 trứng khớp DP minimax với mọi n ≤ 200; n=100 ->", three_egg_drop(100), "| n=1000 ->", three_egg_drop(1000))

Bài 5: 3 trứng khớp DP minimax với mọi n ≤ 200; n=100 -> 9 | n=1000 -> 19


## 20. Template nhớ nhanh + Checklist

```python
# 2 trứng: m nhỏ nhất với m(m+1)/2 >= n
r = math.isqrt(8 * n + 1)
m = (r - 1) // 2
if m * (m + 1) // 2 < n:
    m += 1
return m

# Tổng quát k trứng: F[e] (m lượt) = F[e] (m-1 lượt) + F[e-1] (m-1 lượt) + 1, tăng m tới khi F[k] >= n
```

**Checklist trước khi submit:**
- [ ] Minimax: `max` cho đối thủ (vỡ / không vỡ), `min` cho lựa chọn tầng.
- [ ] Một quả còn lại ⇒ dò tuyến tính (`k` lượt cho `k` tầng).
- [ ] Không dùng nhị phân trên tầng (chỉ đúng với vô số trứng).
- [ ] Làm tròn **lên** khi giải `m(m+1)/2 ≥ n`; dùng `isqrt`.
- [ ] Thử `n = 1` (đáp án 1) và `n = 1000` (đáp án 45).